# Phase 4 — domain adaptation (Kaggle)

Continues fine-tuning the Phase 3 encoder on Swiggy review pairs with
`MultipleNegativesRankingLoss`. Roughly **20 minutes** on a T4.

## Before you run anything

**Two sidebar settings**, same as the Phase 3b ablation:

1. **Accelerator → GPU T4 x2** (or P100).
2. **Internet → On** — needed for `git clone` and `pip install`.

**One dataset.** Click **+ Add Input → Datasets**, and attach the dataset you
created from `echo-phase4.zip`. It must contain `phase4_pairs.jsonl` and an
`encoder/` folder. Cell 1 checks all three and stops with a clear message if any
is missing, so you find out now rather than eight minutes in.

## 1. Check settings and the attached dataset

In [ ]:
import glob, os, socket, sys, torch

gpu = torch.cuda.is_available()
print('GPU:', torch.cuda.get_device_name(0) if gpu else 'NONE')
try:
    socket.create_connection(('huggingface.co', 443), timeout=8); net = True
except OSError:
    net = False
print('Internet:', 'on' if net else 'OFF')

# Print whatever is actually mounted. Kaggle may nest an extracted zip one or
# two levels deep, so searching recursively beats assuming a layout.
print('\n/kaggle/input:')
entries = 0
for root, dirs, files in os.walk('/kaggle/input'):
    depth = root.rstrip('/').count('/') - 2
    if depth > 3:
        dirs[:] = []
        continue
    print('   ' * depth + os.path.basename(root) + '/')
    for f in sorted(files)[:6]:
        print('   ' * (depth + 1) + f)
    if len(files) > 6:
        print('   ' * (depth + 1) + f'... +{len(files) - 6} more')
    entries += 1
if entries <= 1:
    print('   (nothing attached)')

PAIRS = next(iter(glob.glob('/kaggle/input/**/phase4_pairs.jsonl', recursive=True)), None)
cfg = next(iter(glob.glob('/kaggle/input/**/encoder/config.json', recursive=True)), None)
ENCODER = os.path.dirname(cfg) if cfg else None
stray = glob.glob('/kaggle/input/**/*.zip', recursive=True)

print('\nPAIRS  :', PAIRS or 'NOT FOUND')
print('ENCODER:', ENCODER or 'NOT FOUND')

if not gpu: sys.exit('Settings > Accelerator > GPU T4 x2, then re-run.')
if not net: sys.exit('Settings > Internet > On, then re-run.')
if not (PAIRS and ENCODER):
    if stray:
        sys.exit(f'Found an un-extracted zip: {stray[0]}\n'
                 'Kaggle did not expand it. Re-create the dataset and wait for '
                 'the upload to finish processing before clicking Create.')
    sys.exit('Dataset not attached to THIS notebook.\n'
             'Uploading a dataset and attaching it are two separate steps:\n'
             '  right sidebar > + Add Input > Datasets > Your Datasets > click +\n'
             'The tree printed above shows what is currently attached.')

n = sum(1 for _ in open(PAIRS))
print(f'\n{n:,} pairs · encoder found · all checks passed.')

## 2. Get the code

In [ ]:
%cd /kaggle/working
![ -d Echo ] && (cd Echo && git pull -q) || git clone -q https://github.com/ayn-aval/Echo.git
%cd /kaggle/working/Echo
!git log --oneline -1

## 3. Install

`sentence-transformers` is the library Phase 3 was forbidden from using. It is a
thin wrapper over the same HuggingFace pieces we wrote by hand, and it supplies
`MultipleNegativesRankingLoss` ready-made. `accelerate` is not optional — modern
`sentence-transformers` routes `fit()` through the HuggingFace Trainer, which
fails with an `ImportError` without it.

In [ ]:
!pip install -q -U sentence-transformers accelerate 2>&1 | tail -2
import sentence_transformers, accelerate
print('sentence-transformers', sentence_transformers.__version__,
      '| accelerate', accelerate.__version__)

## 4. Train

53,061 pairs: 7,197 mined (two reviews that both TF-IDF and the Phase 3 encoder
independently rank as near-matches) and 45,864 SimCSE self-pairs. See
`src/training/mine_pairs.py` for how they were built, and `eval/reply_signal.py`
for why the reply-based strategies in the project plan were rejected.

`--batch-size 64` matters more than usual: MNRL treats the other 63 items in a
batch as negatives, so a bigger batch is a harder and more informative problem.
Drop it to 32 if you hit CUDA out-of-memory.

In [ ]:
# PAIRS and ENCODER come from cell 1 — re-run it first if the kernel restarted.
assert PAIRS and ENCODER, 'Run cell 1 first.'

cmd = (f"python -m src.training.train_domain"
       f" --pairs {PAIRS}"
       f" --encoder {ENCODER}"
       f" --out /kaggle/working/sbert-domain"
       f" --batch-size 64 --epochs 1 --lr 2e-5")
print(cmd)
!{cmd}

## 5. Sanity check before you download 300 MB

STS is *not* what Phase 4 optimises — the real test is review retrieval, which
needs your local Postgres and runs on the Mac. This is only a smoke test: it
confirms the saved encoder loads and still produces sane vectors.

**Expect STS to drop somewhat.** Phase 3 scored 72.17 average. Specialising on
short, misspelled Hinglish reviews costs generic performance, and that tradeoff
is the finding we report — not a failure to hide.

In [ ]:
from eval.sts_eval import evaluate_sts
from src.embeddings.sbert import make_encoder

enc = make_encoder('/kaggle/working/sbert-domain/encoder')
print(evaluate_sts(enc, 'sbert-domain').to_string(index=False))

## 6. Save the encoder as notebook output

Then **Save Version → Save & Run All**, and download `sbert-domain.zip` from the
notebook's Output tab. Unzip it locally into `models/sbert-domain/` — the eval
harness loads it with no code changes.

In [ ]:
!cd /kaggle/working && zip -qr sbert-domain.zip sbert-domain && ls -lh sbert-domain.zip

## If the session dies

This run is short enough that checkpointing would cost more than it saves — if it
dies, re-run cells 2 onward. `/kaggle/working` survives while the session lives.
For an unattended run use **Save Version → Save & Run All**, which runs detached
and keeps the output permanently.